# Issue #5 — Demonstração: previsão de cancelamento

A equipe de retenção informa os dados de um cliente e recebe a **pontuação de cancelamento** (0 a 1) e a **classe prevista**.

**Para executar:** instale `requirements.txt`, mantenha este notebook na raiz do repositório e rode todas as células em ordem. Não é preciso baixar o CSV nem treinar de novo.

## 1. Carregar os arquivos salvos

A rede e o pré-processador (ajustado **somente no treino**) foram salvos em `artefatos/` pelo `treinamento_modelos.ipynb`. O limiar vem de `configuracao_escolhida.json`.

In [1]:
from previsao import CAMPOS_INTERNET, carregar_artefatos, prever_cancelamento

artefatos = carregar_artefatos()
print("Limiar da rede:", artefatos["limiar"])
print("Campos de entrada:", len(artefatos["campos"]))

Limiar da rede: 0.275
Campos de entrada: 19


## 2. Função de previsão

`prever_cancelamento` (em `previsao.py`) valida os dados, aplica o pré-processador e a rede e devolve a pontuação e a classe.

In [2]:
cliente_exemplo = {
    campo: regra.get("valor_padrao", regra.get("mediana_padrao"))
    for campo, regra in artefatos["schema"].items()
}
resultado = prever_cancelamento(cliente_exemplo, artefatos)
resultado

{'pontuacao': 0.5370433926582336,
 'classe': 'Yes',
 'limiar': 0.275,
 'avisos': []}

## 3. Verificar a previsão após salvar e recarregar

Salvamos a rede e o pré-processador em uma pasta temporária, carregamos tudo de novo e comparamos o resultado.

In [3]:
import shutil
import tempfile
from pathlib import Path

import joblib

with tempfile.TemporaryDirectory() as pasta:
    pasta = Path(pasta)
    artefatos["rede"].save(pasta / "modelo_churn.keras")
    joblib.dump(artefatos["preprocessador"], pasta / "preprocessador.joblib")
    shutil.copy("artefatos/configuracao_escolhida.json", pasta)

    recarregados = carregar_artefatos(pasta)
    resultado_recarregado = prever_cancelamento(cliente_exemplo, recarregados)

assert abs(resultado_recarregado["pontuacao"] - resultado["pontuacao"]) < 1e-6
assert resultado_recarregado["classe"] == resultado["classe"]
print(f"Antes:  {resultado['pontuacao']:.6f} -> {resultado['classe']}")
print(f"Depois: {resultado_recarregado['pontuacao']:.6f} -> {resultado_recarregado['classe']}")
print("OK: a previsão é a mesma depois de recarregar os arquivos.")

Antes:  0.537043 -> Yes
Depois: 0.537043 -> Yes
OK: a previsão é a mesma depois de recarregar os arquivos.


## 4. Tela de previsão

Altere os campos para consultar um cliente. Quando o cliente não tem internet ou telefone, os serviços dependentes são ajustados automaticamente.

In [4]:
import ipywidgets as widgets
from IPython.display import HTML, display

ROTULOS = {
    "gender": "Gênero", "SeniorCitizen": "Idoso (65+)", "Partner": "Tem cônjuge",
    "Dependents": "Tem dependentes", "tenure": "Meses como cliente",
    "PhoneService": "Telefone", "MultipleLines": "Várias linhas",
    "InternetService": "Internet", "OnlineSecurity": "Segurança online",
    "OnlineBackup": "Backup online", "DeviceProtection": "Proteção do aparelho",
    "TechSupport": "Suporte técnico", "StreamingTV": "Streaming de TV",
    "StreamingMovies": "Streaming de filmes", "Contract": "Contrato",
    "PaperlessBilling": "Fatura digital", "PaymentMethod": "Forma de pagamento",
    "MonthlyCharges": "Mensalidade", "TotalCharges": "Total já cobrado",
}
TRADUCOES = {
    "Female": "Feminino", "Male": "Masculino", "Yes": "Sim", "No": "Não",
    "No phone service": "Sem telefone", "No internet service": "Sem internet",
    "Fiber optic": "Fibra óptica", "Month-to-month": "Mensal",
    "One year": "1 ano", "Two year": "2 anos",
    "Bank transfer (automatic)": "Transferência automática",
    "Credit card (automatic)": "Cartão de crédito automático",
    "Electronic check": "Cheque eletrônico", "Mailed check": "Cheque pelo correio",
}


def opcoes(valores):
    return [(TRADUCOES.get(v, v), v) for v in valores]


def criar_campo(campo, regra):
    comum = {"description": ROTULOS[campo], "style": {"description_width": "170px"},
             "layout": widgets.Layout(width="460px")}
    if campo == "SeniorCitizen":
        return widgets.Dropdown(options=[("Não", 0), ("Sim", 1)], value=0, **comum)
    if regra["tipo"] == "categoria":
        return widgets.Dropdown(options=opcoes(regra["valores_aceitos"]),
                                value=regra["valor_padrao"], **comum)
    if campo == "tenure":
        return widgets.IntSlider(min=int(regra["min"]), max=int(regra["max"]),
                                 value=int(regra["mediana_padrao"]),
                                 continuous_update=False, **comum)
    return widgets.BoundedFloatText(min=regra["min"], max=regra["max"],
                                    value=regra["mediana_padrao"], step=0.05, **comum)


campos = {c: criar_campo(c, artefatos["schema"][c]) for c in artefatos["campos"]}


def ligar_dependentes(principal, dependentes, sem_servico):
    """Mantém os campos dependentes coerentes com o serviço principal."""
    def ajustar(_=None):
        desligado = campos[principal].value == "No"
        for nome in dependentes:
            aceitos = artefatos["schema"][nome]["valores_aceitos"]
            validos = [sem_servico] if desligado else [v for v in aceitos if v != sem_servico]
            atual = campos[nome].value
            campos[nome].options = opcoes(validos)  # trocar as opções redefine o valor
            campos[nome].value = atual if atual in validos else validos[0]
            campos[nome].disabled = desligado

    # Registrado antes do interact para ajustar os campos antes de recalcular.
    campos[principal].observe(ajustar, names="value")
    ajustar()


ligar_dependentes("InternetService", CAMPOS_INTERNET, "No internet service")
ligar_dependentes("PhoneService", ["MultipleLines"], "No phone service")


def mostrar_previsao(**cliente):
    try:
        r = prever_cancelamento(cliente, artefatos)
    except ValueError as erro:
        print("Dados inválidos:", erro)
        return
    alto = r["classe"] == "Yes"
    cor = "#b42318" if alto else "#067647"
    titulo = "Risco de cancelamento: priorizar contato" if alto else "Sem alerta de cancelamento"
    numero = lambda x: f"{x:.3f}".replace(".", ",")
    display(HTML(
        f'<div style="border:2px solid {cor};border-radius:8px;padding:10px 14px;width:430px">'
        f'<div style="color:{cor};font-size:17px;font-weight:bold">{titulo}</div>'
        f'<div>Pontuação de cancelamento: <b>{numero(r["pontuacao"])}</b> (0 a 1)</div>'
        f'<div>Classe prevista (Churn): <b>{r["classe"]}</b> — limiar {numero(r["limiar"])}</div>'
        "</div>"
    ))


_ = widgets.interact(mostrar_previsao, **campos)

interactive(children=(Dropdown(description='Gênero', index=1, layout=Layout(width='460px'), options=(('Feminin…